In [1]:
import os
import pandas as pd

In [2]:
# Define paths
lab_events_path = "../data/Mimic-3_patients/LABEVENTS/LABEVENTS_sorted.csv"
note_events_path = "../data/Mimic-3_patients/NOTEEVENTS/NOTEEVENTS_sorted.csv"
output_path = "../processed/labs_notes_merged.csv"

In [3]:
# Load data
df_lab = pd.read_csv(lab_events_path, usecols=["SUBJECT_ID", "HADM_ID", "ITEMID", "CHARTTIME", "VALUE", "VALUENUM"])
df_notes = pd.read_csv(note_events_path, usecols=["SUBJECT_ID", "HADM_ID", "CHARTTIME", "CATEGORY", "TEXT"])


In [4]:
# Ensure datetime
df_lab["CHARTTIME"] = pd.to_datetime(df_lab["CHARTTIME"], errors="coerce")
df_notes["CHARTTIME"] = pd.to_datetime(df_notes["CHARTTIME"], errors="coerce")


In [5]:
# Filter discharge summaries
df_notes = df_notes[df_notes["CATEGORY"] == "Discharge summary"]


In [6]:
# Merge lab events and notes on SUBJECT_ID and HADM_ID
df_merged = pd.merge(df_lab, df_notes, on=["SUBJECT_ID", "HADM_ID"], suffixes=("_lab", "_note"))


In [7]:
# Calculate time difference
df_merged["time_diff"] = (df_merged["CHARTTIME_note"] - df_merged["CHARTTIME_lab"]).dt.total_seconds() / 3600


In [8]:
# Filter for lab events within 48h before the discharge summary
df_final = df_merged[(df_merged["time_diff"] >= 0) & (df_merged["time_diff"] <= 48)]


In [9]:
# Save result
os.makedirs("../processed", exist_ok=True)
df_final.to_csv(output_path, index=False)


In [12]:
# Exibe as primeiras linhas do dataframe
print("Laboratórios e Notas Integradas (48h antes da alta):")
print(df_final.head())

# E salve o resultado
df_final.to_csv("../processed/labs_notas_integradas.csv", index=False)


Laboratórios e Notas Integradas (48h antes da alta):
Empty DataFrame
Columns: [SUBJECT_ID, HADM_ID, ITEMID, CHARTTIME_lab, VALUE, VALUENUM, CHARTTIME_note, CATEGORY, TEXT, time_diff]
Index: []
